# JEPA World Model — Colab training + eval

Runtime: **GPU (H100 or A100)**. Expected end-to-end time: 30–60 min on H100 for data_2k @ 30 epochs.

Before running:
1. Upload `data_500/` (or `data_2k/`) to Drive at `MyDrive/optical_wm/`.
2. Set Runtime → Change runtime type → H100 (or A100).
3. Run cells top to bottom.

In [ ]:
# 1. GPU sanity check
!nvidia-smi | head -20

In [ ]:
# 2. Clone repo and install
!git clone https://github.com/Anishanks/optical-network-wm.git /content/repo
%cd /content/repo
!git log -1 --oneline
!pip install -q torch h5py numpy networkx matplotlib
!pip install -q -e .

In [ ]:
# 3. Mount Drive and copy data to local disk (local I/O is ~10× faster than Drive FUSE)
from google.colab import drive
drive.mount('/content/drive')

DATA_NAME = 'data_2k'     # or 'data_500' / 'data_full'
DRIVE_PATH = f'/content/drive/MyDrive/optical_wm/{DATA_NAME}'
LOCAL_DATA = f'/content/repo/{DATA_NAME}'

!cp -r '{DRIVE_PATH}' '{LOCAL_DATA}'
!ls -la '{LOCAL_DATA}'

In [ ]:
# 4. Train (tune these knobs)
RUN_NAME = 'colab_2k_ctx16'
CTX = 16          # context length; max rollout horizon = CTX - 1
EPOCHS = 40
BATCH_SIZE = 64   # H100 can go to 128+; reduce if OOM
LR = 3e-4
COLLAPSE_WEIGHT = 0.1

import os
os.environ['PYTHONPATH'] = '/content/repo/src'

!python -m optical_wm.train \
  --data {LOCAL_DATA} \
  --name {RUN_NAME} \
  --context-length {CTX} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR} \
  --collapse-weight {COLLAPSE_WEIGHT} \
  --num-workers 2 \
  --device cuda \
  --output /content/repo/checkpoints

In [ ]:
# 5. Probing (predictive: z_t -> gf_{t+k})
CKPT = f'/content/repo/checkpoints/{RUN_NAME}/best.pt'
PROBE_OUT = f'/content/repo/figures_{DATA_NAME}/probing_{RUN_NAME}'
HORIZONS_PROBE = '1 3 5 10 15'

!python -m optical_wm.evaluation.probing \
  --checkpoint {CKPT} \
  --data {LOCAL_DATA} \
  --output {PROBE_OUT} \
  --horizons {HORIZONS_PROBE} \
  --probe-epochs 100 \
  --batch-size 32 \
  --device cuda

In [ ]:
# 6. Rollout (JEPA vs direct-linear vs persistence, in embedding space)
ROLL_OUT = f'/content/repo/figures_{DATA_NAME}/rollout_{RUN_NAME}'
MAX_HORIZON = CTX - 1

!python -m optical_wm.evaluation.rollout \
  --checkpoint {CKPT} \
  --data {LOCAL_DATA} \
  --output {ROLL_OUT} \
  --max-horizon {MAX_HORIZON} \
  --batch-size 32 \
  --no-probing \
  --device cuda

In [ ]:
# 7. Push results + checkpoint back to Drive
DRIVE_OUT = f'/content/drive/MyDrive/optical_wm/runs/{RUN_NAME}'
!mkdir -p '{DRIVE_OUT}'
!cp -r /content/repo/checkpoints/{RUN_NAME} '{DRIVE_OUT}/checkpoints'
!cp -r /content/repo/figures_{DATA_NAME} '{DRIVE_OUT}/figures'
!ls -la '{DRIVE_OUT}'

In [ ]:
# 8. Inline preview of the key plots
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'/content/repo/figures_{DATA_NAME}/**/*.png', recursive=True)):
    print(p); display(Image(p))